In [0]:

# NOTEBOOK 04 — Graph-Augmented K-Means Risk Clustering
# Uses PageRank centrality features from Notebook 03.
# Runs TWO models: transactional only vs graph-augmented.
# Compares silhouette scores to prove graph features add value.


from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import mlflow, mlflow.spark, time

STORAGE_ACCOUNT     = "adlssupplychain"
CONTAINER           = "supplychain-data"
BASE_ABFSS          = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
GOLD_CLUSTERS_TRANS = f"{BASE_ABFSS}/gold/risk_clusters_transactional"
GOLD_CLUSTERS_GRAPH = f"{BASE_ABFSS}/gold/risk_clusters_graph_augmented"
GOLD_PAGERANK_ABFSS = f"{BASE_ABFSS}/gold/pagerank_results"

mlflow.set_experiment("/supply-chain-pipeline")


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1418298044073275', creation_time=1776371864602, experiment_id='1418298044073275', last_update_time=1776506746839, lifecycle_stage='active', name='/supply-chain-pipeline', tags={'mlflow.experiment.sourceName': '/supply-chain-pipeline',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'sudheeshudhayakumar07@gmail.com',
 'mlflow.ownerId': '145444312514419'}>

In [0]:

# SECTION 0 — Global config + clean slate


spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set(
    "spark.databricks.delta.overwriteSchema.enabled",  "true")

for tbl in [
    "supply_chain_db.risk_clusters_transactional",
    "supply_chain_db.risk_clusters_graph_augmented",
]:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")

for path in [GOLD_CLUSTERS_TRANS, GOLD_CLUSTERS_GRAPH]:
    try:
        dbutils.fs.rm(path, recurse=True)
        print(f"  Cleared: {path.split('/')[-1]}")
    except Exception:
        print(f"  Already empty: {path.split('/')[-1]}")

spark.sql("CLEAR CACHE")
spark.catalog.clearCache()
print("Clean slate confirmed.\n")


  Cleared: risk_clusters_transactional
  Cleared: risk_clusters_graph_augmented
Clean slate confirmed.



In [0]:

# SECTION 1 — Load data

print("Loading silver table...")
df = (spark.table("supply_chain_db.supply_chain_100gb")
      .withColumn("Order_Region",
          F.trim(F.col("Order_Region")))
      .withColumn("Market",
          F.trim(F.col("Market")))
      .withColumn("Order_Country",
          F.trim(F.col("Order_Country")))
     )

print("Loading PageRank results from Notebook 03...")
pr_df = (spark.read.format("delta")
         .load(GOLD_PAGERANK_ABFSS)
         .select(
             F.col("id").alias("hub_id"),
             F.col("pagerank").alias("hub_pagerank"),
             F.col("avg_delay_risk").alias("hub_avg_delay_risk"),
             F.col("order_count").alias("hub_order_count"),
             F.col("pagerank_raw").alias("hub_pagerank_raw"),
         )
)

hub_count = pr_df.count()
print(f"Hub nodes loaded: {hub_count:,}")

print("\nPageRank distribution (verify not all 1.0):")
pr_df.select(
    F.min("hub_pagerank").alias("min"),
    F.max("hub_pagerank").alias("max"),
    F.countDistinct("hub_pagerank").alias("distinct_values"),
).show()

print("\nTop 10 bottleneck hubs:")
display(
    pr_df.orderBy(F.col("hub_pagerank").desc()).limit(10))


Loading silver table...
Loading PageRank results from Notebook 03...
Hub nodes loaded: 167

PageRank distribution (verify not all 1.0):
+---+---+---------------+
|min|max|distinct_values|
+---+---+---------------+
|0.0|1.0|            167|
+---+---+---------------+


Top 10 bottleneck hubs:


hub_id,hub_pagerank,hub_avg_delay_risk,hub_order_count,hub_pagerank_raw
LATAM|Central America|México,1.0,0.53751814656405,7476622,2.4377143047696572
LATAM|South America|Brasil,0.978595293903318,0.529722136680857,4539714,2.3890421753688513
Europe|Western Europe|Francia,0.9681680324922367,0.5394404047746046,7448491,2.365331641133819
Europe|Northern Europe|Reino Unido,0.9023603178050292,0.5230932145076639,4141866,2.2156915764508667
Pacific Asia|Oceania|Australia,0.8808040742672117,0.5284736558084157,4777258,2.1666748662877415
USCA|West of USA|Estados Unidos,0.8721924644763506,0.5236374387681982,4495009,2.147092939575138
Europe|Western Europe|Alemania,0.8634588565311925,0.5503467264071025,5399358,2.127233601440683
USCA|East of USA|Estados Unidos,0.85362887078945,0.5398261976176502,3888094,2.104881211463026
Europe|Southern Europe|Italia,0.8354975342127954,0.5182654170948717,2774177,2.06365239215187
LATAM|Caribbean|República Dominicana,0.8326094826331977,0.5368016736618851,2080707,2.0570852559546497


In [0]:

# SECTION 2 — Join PageRank onto silver table

print("\nJoining PageRank scores onto orders...")

df_with_hub = df.withColumn(
    "hub_id",
    F.concat_ws("|",
        F.col("Market"),
        F.col("Order_Region"),
        F.col("Order_Country"))
)

df_enriched = (df_with_hub
    .join(pr_df, on="hub_id", how="left")
    .fillna({
        "hub_pagerank":       0.0,
        "hub_avg_delay_risk": 0.0,
        "hub_order_count":    0,
        "hub_pagerank_raw":   0.0,
    })
)

# Verify join worked
join_check = (df_enriched
    .filter(F.col("hub_pagerank") > 0)
    .count()
)
total_rows = df_enriched.count()
print(f"Rows with PageRank score > 0: {join_check:,} "
      f"of {total_rows:,} "
      f"({100*join_check/total_rows:.1f}%)")

print("\nSample — hub_pagerank values after join:")
(df_enriched
    .select("Market", "Order_Region", "hub_id",
            "hub_pagerank", "hub_avg_delay_risk")
    .distinct()
    .orderBy(F.col("hub_pagerank").desc())
    .show(10, truncate=False))



Joining PageRank scores onto orders...
Rows with PageRank score > 0: 101,839,253 of 101,840,453 (100.0%)

Sample — hub_pagerank values after join:
+------------+---------------+------------------------------------+------------------+------------------+
|Market      |Order_Region   |hub_id                              |hub_pagerank      |hub_avg_delay_risk|
+------------+---------------+------------------------------------+------------------+------------------+
|LATAM       |Central America|LATAM|Central America|México        |1.0               |0.53751814656405  |
|LATAM       |South America  |LATAM|South America|Brasil          |0.978595293903318 |0.529722136680857 |
|Europe      |Western Europe |Europe|Western Europe|Francia       |0.9681680324922367|0.5394404047746046|
|Europe      |Northern Europe|Europe|Northern Europe|Reino Unido  |0.9023603178050292|0.5230932145076639|
|Pacific Asia|Oceania        |Pacific Asia|Oceania|Australia      |0.8808040742672117|0.5284736558084157|
|USC

In [0]:

# SECTION 3 — Feature aggregation
# Finer groupBy: Market + Order_Region + Order_Country


print("\nAggregating features per segment...")
t_agg = time.time()

feature_df = (df_enriched
    .groupBy(
        "Order_Region",
        "Market",
        "Order_Country",
        "Shipping_Mode",
    )
    .agg(
        #  Transactional features 
        F.avg("Late_delivery_risk")
         .alias("avg_delay_risk"),
        F.avg("Order_Item_Discount_Rate")
         .alias("avg_discount_rate"),
        F.avg("Order_Profit_Per_Order")
         .alias("avg_profit"),
        F.avg("Sales")
         .alias("avg_sales"),
        F.count("Order_Id")
         .alias("order_volume"),
        F.avg("Days_for_shipping_real")
         .alias("avg_real_ship_days"),
        F.avg("Days_for_shipment_scheduled")
         .alias("avg_sched_ship_days"),
        F.avg("Order_Item_Profit_Ratio")
         .alias("avg_profit_ratio"),
        F.avg("Benefit_per_order")
         .alias("avg_benefit"),
        F.avg(
            F.col("Days_for_shipping_real") -
            F.col("Days_for_shipment_scheduled")
        ).alias("avg_delay_gap"),

        #  Graph-derived features (novel contribution) ─
        F.avg("hub_pagerank")
         .alias("avg_hub_pagerank"),
        F.max("hub_pagerank")
         .alias("max_hub_pagerank"),
        F.avg("hub_avg_delay_risk")
         .alias("avg_hub_network_risk"),
        F.avg("hub_order_count")
         .alias("avg_hub_order_count"),
        # Centrality-weighted risk:
        # high value = segment routes through high-centrality
        # hubs AND has high delay risk = maximum systemic exposure
        F.avg(
            F.col("hub_pagerank") *
            F.col("Late_delivery_risk")
        ).alias("centrality_weighted_risk"),
    )
    .na.drop()
).cache()

agg_time       = time.time() - t_agg
total_segments = feature_df.count()

print(f"Aggregation complete in {agg_time:.0f}s")
print(f"Total segments: {total_segments:,}")
print(f"  (Market × Order_Region × Order_Country × Shipping_Mode)")

print("\nSample — graph features in aggregated data:")
feature_df.select(
    "Order_Region", "Market", "Shipping_Mode",
    F.round("avg_hub_pagerank", 4).alias("hub_centrality"),
    F.round("centrality_weighted_risk", 4).alias("centrality_risk"),
    F.round("avg_delay_risk", 3).alias("delay_risk"),
).orderBy(F.col("avg_hub_pagerank").desc()).show(10)



Aggregating features per segment...
Aggregation complete in 1s
Total segments: 561
  (Market × Order_Region × Order_Country × Shipping_Mode)

Sample — graph features in aggregated data:
+---------------+------+--------------+--------------+---------------+----------+
|   Order_Region|Market| Shipping_Mode|hub_centrality|centrality_risk|delay_risk|
+---------------+------+--------------+--------------+---------------+----------+
|Central America| LATAM|  Second Class|           1.0|         0.7299|      0.73|
|Central America| LATAM|Standard Class|           1.0|         0.3963|     0.396|
|Central America| LATAM|   First Class|           1.0|         0.9103|      0.91|
|Central America| LATAM|      Same Day|           1.0|         0.0399|      0.04|
|  South America| LATAM|      Same Day|        0.9786|         0.0391|      0.04|
|  South America| LATAM|  Second Class|        0.9786|         0.7237|     0.739|
|  South America| LATAM|   First Class|        0.9786|         0.8735|     

In [0]:
# SECTION 4 — Define feature sets
# Model A: transactional only (baseline)
# Model B: transactional + graph features (novel)

TRANSACTIONAL_FEATURES = [
    "avg_delay_risk",
    "avg_discount_rate",
    "avg_profit",
    "avg_sales",
    "order_volume",
    "avg_real_ship_days",
    "avg_delay_gap",
    "avg_profit_ratio",
    "avg_benefit",
]

GRAPH_AUGMENTED_FEATURES = TRANSACTIONAL_FEATURES + [
    "avg_hub_pagerank",
    "max_hub_pagerank",
    "avg_hub_network_risk",
    "centrality_weighted_risk",
]

print(f"\nModel A — transactional only  : "
      f"{len(TRANSACTIONAL_FEATURES)} features")
print(f"Model B — graph-augmented     : "
      f"{len(GRAPH_AUGMENTED_FEATURES)} features")
print(f"Graph features added          : "
      f"{len(GRAPH_AUGMENTED_FEATURES)-len(TRANSACTIONAL_FEATURES)}")

evaluator = ClusteringEvaluator(
    featuresCol="features", metricName="silhouette")



Model A — transactional only  : 9 features
Model B — graph-augmented     : 13 features
Graph features added          : 4


In [0]:

# SECTION 5 — Train both models

model_configs = [
    ("transactional_only",    TRANSACTIONAL_FEATURES,
     GOLD_CLUSTERS_TRANS,
     "supply_chain_db.risk_clusters_transactional"),
    ("graph_augmented",       GRAPH_AUGMENTED_FEATURES,
     GOLD_CLUSTERS_GRAPH,
     "supply_chain_db.risk_clusters_graph_augmented"),
]

results = {}

for model_name, feature_cols, save_path, table_name \
        in model_configs:

    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"Features: {len(feature_cols)}")
    print(f"{'='*60}")

    best_k, best_score     = 2, -1.0
    best_model, best_preds = None, None

    with mlflow.start_run(run_name=f"kmeans_{model_name}"):
        mlflow.set_tag("team_member",  "Chinmaye")
        mlflow.set_tag("model_type",   model_name)
        mlflow.set_tag("compute_type", "Standard_D4s_v3")
        mlflow.log_param("feature_cols",  str(feature_cols))
        mlflow.log_param("n_features",    len(feature_cols))
        mlflow.log_param("n_segments",    total_segments)

        #  Elbow search k=2 to 8 ─
        print("\nElbow search (k=2 to 8):")
        for k in range(2, 9):
            with mlflow.start_run(
                    run_name=f"kmeans_{model_name}_k{k}",
                    nested=True):
                mlflow.log_param("k",          k)
                mlflow.log_param("model_type", model_name)

                asm  = VectorAssembler(
                           inputCols=feature_cols,
                           outputCol="raw_f",
                           handleInvalid="skip")
                scl  = StandardScaler(
                           inputCol="raw_f",
                           outputCol="features",
                           withStd=True, withMean=True)
                km   = KMeans(
                           k=k, seed=42,
                           featuresCol="features",
                           maxIter=50, tol=1e-4)
                pipe = Pipeline(stages=[asm, scl, km])

                t0      = time.time()
                mdl     = pipe.fit(feature_df)
                preds   = mdl.transform(feature_df)
                score   = evaluator.evaluate(preds)
                elapsed = time.time() - t0

                mlflow.log_metric("silhouette",      score)
                mlflow.log_metric("train_time_sec",  elapsed)
                print(f"  k={k}  silhouette={score:.4f}"
                      f"  ({elapsed:.0f}s)")

                if score > best_score:
                    best_score = score
                    best_k     = k
                    best_model = mdl
                    best_preds = preds

        mlflow.log_param("best_k",           best_k)
        mlflow.log_metric("best_silhouette", best_score)
        mlflow.spark.log_model(
            best_model, f"kmeans_{model_name}_k{best_k}")

        print(f"\n  Best k={best_k}  "
              f"silhouette={best_score:.4f}")

        #  Cluster profiles ─
        print(f"\nCluster profiles:")
        if model_name == "graph_augmented":
            display(
                best_preds
                .groupBy("prediction")
                .agg(
                    F.avg("avg_delay_risk")
                     .alias("mean_delay_risk"),
                    F.avg("avg_hub_pagerank")
                     .alias("mean_hub_centrality"),
                    F.avg("centrality_weighted_risk")
                     .alias("mean_centrality_risk"),
                    F.avg("avg_sales")
                     .alias("mean_sales"),
                    F.avg("avg_profit")
                     .alias("mean_profit"),
                    F.avg("avg_delay_gap")
                     .alias("mean_delay_gap"),
                    F.count("*")
                     .alias("segment_count"),
                )
                .orderBy("mean_delay_risk")
            )
        else:
            display(
                best_preds
                .groupBy("prediction")
                .agg(
                    F.avg("avg_delay_risk")
                     .alias("mean_delay_risk"),
                    F.avg("avg_sales")
                     .alias("mean_sales"),
                    F.avg("avg_profit")
                     .alias("mean_profit"),
                    F.avg("avg_delay_gap")
                     .alias("mean_delay_gap"),
                    F.avg("avg_real_ship_days")
                     .alias("mean_ship_days"),
                    F.count("*")
                     .alias("segment_count"),
                )
                .orderBy("mean_delay_risk")
            )

        #  Save to Delta 
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")
        (best_preds.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("mergeSchema",     "true")
            .option("path", save_path)
            .saveAsTable(table_name))
        print(f"\nSaved to {table_name}")

        results[model_name] = {
            "best_k":      best_k,
            "silhouette":  best_score,
            "predictions": best_preds,
            "n_features":  len(feature_cols),
        }



Training: transactional_only
Features: 9

Elbow search (k=2 to 8):
  k=2  silhouette=0.4504  (27s)
  k=3  silhouette=0.4349  (11s)
  k=4  silhouette=0.3643  (17s)
  k=5  silhouette=0.3070  (17s)
  k=6  silhouette=0.3443  (13s)
  k=7  silhouette=0.4040  (14s)
  k=8  silhouette=0.4255  (13s)


2026/04/18 10:37:29 INFO mlflow.spark: Inferring pip requirements by reloading the logged model from the databricks artifact repository, which can be time-consuming. To speed up, explicitly specify the conda_env or pip_requirements when calling log_model().
2026/04/18 10:38:34 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2023-07-14; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'databricks-feature-engineering'}
/databricks/python/lib/python3.10/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")



  Best k=2  silhouette=0.4504

Cluster profiles:


prediction,mean_delay_risk,mean_sales,mean_profit,mean_delay_gap,mean_ship_days,segment_count
0,0.05129909656847263,206.2771937755706,34.30142222178356,-0.11841003356794529,0.45303229458779826,133
1,0.7105454660126148,198.78228254008243,34.31072514264198,1.057228622010846,3.391347039900214,428



Saved to supply_chain_db.risk_clusters_transactional

Training: graph_augmented
Features: 13

Elbow search (k=2 to 8):
  k=2  silhouette=0.2030  (20s)
  k=3  silhouette=0.2863  (13s)
  k=4  silhouette=0.3229  (18s)
  k=5  silhouette=0.3159  (19s)
  k=6  silhouette=0.3009  (27s)
  k=7  silhouette=0.3290  (16s)
  k=8  silhouette=0.2662  (21s)


2026/04/18 10:41:21 INFO mlflow.spark: Inferring pip requirements by reloading the logged model from the databricks artifact repository, which can be time-consuming. To speed up, explicitly specify the conda_env or pip_requirements when calling log_model().
2026/04/18 10:42:20 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2023-07-14; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'databricks-feature-engineering'}



  Best k=7  silhouette=0.3290

Cluster profiles:


prediction,mean_delay_risk,mean_hub_centrality,mean_centrality_risk,mean_sales,mean_profit,mean_delay_gap,segment_count
6,0.04012533870211035,0.5526155724395823,0.022122464171388703,192.72213898612574,34.403251524782135,0.0,83
1,0.2682959325396826,0.23779277119397296,0.03820823574076075,317.2605266296503,94.643341800581,0.4843974452532077,20
5,0.3594249011410053,0.3384146211113957,0.13077890664071776,197.40913470706698,34.62533241434999,-0.09750290496101846,120
3,0.41580552352833544,0.8409576615241278,0.35397196629024313,203.7146769061261,34.6809609686626,0.15955026442892017,25
2,0.4564209221922337,0.21721960744283694,0.07690521457400887,200.9901480840176,-6.737467758102742,0.65623748212946,44
4,0.8380484200437592,0.6893371325081816,0.5768606195588574,200.93986212372442,33.839271614977385,1.450246538044813,102
0,0.8571876160951285,0.26756499469840916,0.22608321176135684,191.92299455716707,37.85342375251322,1.5447267230022335,167



Saved to supply_chain_db.risk_clusters_graph_augmented


In [0]:

# SECTION 6 — Comparison

sil_trans  = results["transactional_only"]["silhouette"]
sil_graph  = results["graph_augmented"]["silhouette"]
improvement = ((sil_graph - sil_trans) / abs(sil_trans)) * 100

print("\n" + "="*60)
print("COMPARISON RESULT")
print("="*60)
print(f"\n  Model A — Transactional only")
print(f"    Features  : {results['transactional_only']['n_features']}")
print(f"    Best k    : {results['transactional_only']['best_k']}")
print(f"    Silhouette: {sil_trans:.4f}")
print(f"\n  Model B — Graph-augmented")
print(f"    Features  : {results['graph_augmented']['n_features']}")
print(f"    Best k    : {results['graph_augmented']['best_k']}")
print(f"    Silhouette: {sil_graph:.4f}")
print(f"\n  Improvement: {improvement:+.2f}%")

with mlflow.start_run(run_name="clustering_comparison"):
    mlflow.log_metric("silhouette_transactional",   sil_trans)
    mlflow.log_metric("silhouette_graph_augmented", sil_graph)
    mlflow.log_metric("silhouette_improvement_pct", improvement)
    mlflow.log_param("best_k_transactional",
                     results["transactional_only"]["best_k"])
    mlflow.log_param("best_k_graph_augmented",
                     results["graph_augmented"]["best_k"])
    mlflow.set_tag("finding",
                   "improved" if sil_graph > sil_trans
                   else "no_improvement")

if sil_graph > sil_trans:
    print()
    print("  FINDING: Graph features IMPROVE cluster quality.")
    print()
    print("  Thesis statement:")
    print("  'Incorporating PageRank centrality features from")
    print("   GraphFrames into the K-Means feature space produces")
    print(f"  a {improvement:.1f}% improvement in silhouette score,")
    print("   demonstrating that supply chain network topology")
    print("   carries discriminative information beyond what")
    print("   transactional records alone can reveal.'")
else:
    diff = abs(improvement)
    print()
    print("  FINDING: Graph features did not improve silhouette.")
    print()
    print("  Thesis statement:")
    print("  'Graph centrality features did not improve cluster")
    print(f"  separation ({diff:.1f}% change), establishing that")
    print("   transactional patterns already capture the variance")
    print("   that network position contributes for this dataset.")
    print("   This empirically bounds the utility of graph")
    print("   augmentation in supply chain risk segmentation.'")

print()
print("Both outcomes are valid and publishable findings.")



COMPARISON RESULT

  Model A — Transactional only
    Features  : 9
    Best k    : 2
    Silhouette: 0.4504

  Model B — Graph-augmented
    Features  : 13
    Best k    : 7
    Silhouette: 0.3290

  Improvement: -26.96%

  FINDING: Graph features did not improve silhouette.

  Thesis statement:
  'Graph centrality features did not improve cluster
  separation (27.0% change), establishing that
   transactional patterns already capture the variance
   that network position contributes for this dataset.
   This empirically bounds the utility of graph
   augmentation in supply chain risk segmentation.'

Both outcomes are valid and publishable findings.


In [0]:

# SECTION 7 — Cluster interpretation
print("\n" + "="*60)
print("GRAPH-AUGMENTED CLUSTER INTERPRETATION")
print("="*60)

cluster_profiles = (results["graph_augmented"]["predictions"]
    .groupBy("prediction")
    .agg(
        F.avg("avg_delay_risk").alias("mean_delay_risk"),
        F.avg("avg_hub_pagerank").alias("mean_hub_centrality"),
        F.avg("centrality_weighted_risk")
         .alias("mean_centrality_risk"),
        F.avg("avg_sales").alias("mean_sales"),
        F.avg("avg_profit").alias("mean_profit"),
        F.avg("avg_delay_gap").alias("mean_delay_gap"),
        F.avg("avg_real_ship_days").alias("mean_ship_days"),
        F.count("*").alias("segment_count"),
    )
    .orderBy("mean_delay_risk")
    .toPandas()
)

print("\nCluster profiles (ordered low → high risk):")
print(cluster_profiles.to_string(index=False))

median_centrality = cluster_profiles["mean_hub_centrality"].median()

print("\nInterpretation:")
for _, row in cluster_profiles.iterrows():
    c   = int(row["prediction"])
    dr  = row["mean_delay_risk"]
    pr  = row["mean_hub_centrality"]
    cwr = row["mean_centrality_risk"]
    n   = int(row["segment_count"])

    risk_label = (
        "LOW RISK"    if dr < 0.3 else
        "MEDIUM RISK" if dr < 0.6 else
        "HIGH RISK"
    )
    network_label = (
        "high-centrality (bottleneck) hubs"
        if pr > median_centrality
        else "peripheral hubs"
    )

    print(f"  Cluster {c} [{risk_label:11s}]: "
          f"delay={dr:.3f} | "
          f"centrality={pr:.4f} ({network_label}) | "
          f"n={n:,}")

feature_df.unpersist()
print("\nNotebook 04 complete. Proceed to Notebook 05.")


GRAPH-AUGMENTED CLUSTER INTERPRETATION

Cluster profiles (ordered low → high risk):
 prediction  mean_delay_risk  mean_hub_centrality  mean_centrality_risk  mean_sales  mean_profit  mean_delay_gap  mean_ship_days  segment_count
          6         0.040125             0.552616              0.022122  192.722139    34.403252        0.000000        0.000000             83
          1         0.268296             0.237793              0.038208  317.260527    94.643342        0.484397        1.284609             20
          5         0.359425             0.338415              0.130779  197.409135    34.625332       -0.097503        3.869188            120
          3         0.415806             0.840958              0.353972  203.714677    34.680961        0.159550        3.999586             25
          2         0.456421             0.217220              0.076905  200.990148    -6.737468        0.656237        1.883390             44
          4         0.838048             0.689337  